In [26]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# --- CONFIGURATION ---
# These parameters MUST match the ones used in the training/generation scripts.
SEQ_LEN = 48      # Input history length
PRED_LEN = 12     # Output forecast horizon
DATA_PATH = '../../../ICL4DT/data/time_series_datasets/ETTm2.csv'
HTI_DATA_DIR = 'hti_data'

# The quantiles of the expert models you want to visualize
QUANTILES_TO_LOAD = [0.1, 0.25, 0.5, 0.75,0.9]

print("Loading original dataset...")
df = pd.read_csv(DATA_PATH)
data = df['OT'].values.astype(float)

print(f"Full dataset shape: {data.shape}")

# Recreate the exact train/val/test split to fit the scaler correctly
train_split_idx = int(len(data) * 0.7)
val_split_idx = int(len(data) * 0.98)

# Isolate the original, unscaled test data for ground truth comparison
original_test_data = data[val_split_idx:]

# Fit the scaler ONLY on the training data to prevent data leakage
print("Fitting MinMaxScaler on the training data portion...")
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(data[:train_split_idx].reshape(-1, 1))

hti_datasets = {}
all_forecasts_unscaled = {}

print("Loading HTI datasets and unscaling forecasts...")

for q in QUANTILES_TO_LOAD:
    # Construct filename (e.g., hti_data_q05.pt)
    filename = f"hti_data_q{str(q).replace('.', '')}.pt"
    
    try:
        # Load the entire [history, forecast] tensor
        hti_datasets[q] = torch.load(filename)
        data = hti_datasets[q]
        print(f" -> Loaded '{filename}' with shape: {hti_datasets[q].shape}")
        data_unscaled = scaler.inverse_transform(data)
        data_unscaled = torch.tensor(data_unscaled, dtype=torch.float32)
        all_forecasts_unscaled[q] = data_unscaled
        
    except FileNotFoundError:
        print(f" -> WARNING: Could not find file {filename}. Skipping.")

print("\nForecasts are now unscaled and ready for plotting.")


Loading original dataset...
Full dataset shape: (69680,)
Fitting MinMaxScaler on the training data portion...
Loading HTI datasets and unscaling forecasts...
 -> Loaded 'hti_data_q01.pt' with shape: torch.Size([1335, 60])
 -> Loaded 'hti_data_q025.pt' with shape: torch.Size([1335, 60])
 -> Loaded 'hti_data_q05.pt' with shape: torch.Size([1335, 60])
 -> Loaded 'hti_data_q075.pt' with shape: torch.Size([1335, 60])
 -> Loaded 'hti_data_q09.pt' with shape: torch.Size([1335, 60])

Forecasts are now unscaled and ready for plotting.


In [27]:
all_forecasts_unscaled

{0.1: tensor([[34.1700, 34.3895, 35.0485,  ..., 39.8413, 39.5848, 39.3952],
         [34.3895, 35.0485, 35.4885,  ..., 39.1778, 38.9171, 38.7244],
         [35.0485, 35.4885, 36.1475,  ..., 39.0228, 38.7823, 38.6103],
         ...,
         [39.8340, 39.8340, 39.6145,  ..., 44.2962, 44.0768, 43.9166],
         [39.8340, 39.6145, 39.6145,  ..., 44.2641, 44.0409, 43.8781],
         [39.6145, 39.6145, 39.6145,  ..., 44.2278, 44.0008, 43.8355]]),
 0.25: tensor([[34.1700, 34.3895, 35.0485,  ..., 40.6384, 40.3763, 40.1362],
         [34.3895, 35.0485, 35.4885,  ..., 40.0101, 39.7600, 39.5348],
         [35.0485, 35.4885, 36.1475,  ..., 39.7871, 39.5554, 39.3485],
         ...,
         [39.8340, 39.8340, 39.6145,  ..., 45.5971, 45.3691, 45.2171],
         [39.8340, 39.6145, 39.6145,  ..., 45.5961, 45.3623, 45.2002],
         [39.6145, 39.6145, 39.6145,  ..., 45.6237, 45.3894, 45.2255]]),
 0.5: tensor([[34.1700, 34.3895, 35.0485,  ..., 40.6477, 40.3752, 40.1462],
         [34.3895, 35.0485, 3

In [28]:
combined = torch.stack([all_forecasts_unscaled[q] for q in QUANTILES_TO_LOAD], dim=0)

torch.save(combined, 'hti_data_combined.pt')

In [29]:
combined.shape

torch.Size([5, 1335, 60])

In [30]:
combined

tensor([[[34.1700, 34.3895, 35.0485,  ..., 39.8413, 39.5848, 39.3952],
         [34.3895, 35.0485, 35.4885,  ..., 39.1778, 38.9171, 38.7244],
         [35.0485, 35.4885, 36.1475,  ..., 39.0228, 38.7823, 38.6103],
         ...,
         [39.8340, 39.8340, 39.6145,  ..., 44.2962, 44.0768, 43.9166],
         [39.8340, 39.6145, 39.6145,  ..., 44.2641, 44.0409, 43.8781],
         [39.6145, 39.6145, 39.6145,  ..., 44.2278, 44.0008, 43.8355]],

        [[34.1700, 34.3895, 35.0485,  ..., 40.6384, 40.3763, 40.1362],
         [34.3895, 35.0485, 35.4885,  ..., 40.0101, 39.7600, 39.5348],
         [35.0485, 35.4885, 36.1475,  ..., 39.7871, 39.5554, 39.3485],
         ...,
         [39.8340, 39.8340, 39.6145,  ..., 45.5971, 45.3691, 45.2171],
         [39.8340, 39.6145, 39.6145,  ..., 45.5961, 45.3623, 45.2002],
         [39.6145, 39.6145, 39.6145,  ..., 45.6237, 45.3894, 45.2255]],

        [[34.1700, 34.3895, 35.0485,  ..., 40.6477, 40.3752, 40.1462],
         [34.3895, 35.0485, 35.4885,  ..., 39

In [31]:
combined_min = combined.min().item()
combined_max = combined.max().item()
print(f"Min value in combined: {combined_min}")
print(f"Max value in combined: {combined_max}")

Min value in combined: 22.18130874633789
Max value in combined: 53.98412322998047
